In [ ]:
import mysql.connector                 # Importamos las librarias necesarias
from mysql.connector import errorcode

In [ ]:
try:
    # nos conectamos al servidor de mysql, no especificamos schema porque vamos a crearlo
    
    cnx = mysql.connector.connect(
        user='root',
        password= input('password_MySQL'),
        host='127.0.0.1'
        # auth_plugin = 'mysql_native_password'
    )
    print("¡Conexión exitosa a MySQL!") #
except mysql.connector.Error as err:
    # Si es un error de acceso denegado (ej. contraseña o usuario incorrecto)
    if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
        print("Algo está mal con tu nombre de usuario o contraseña.") #
    # Si la base de datos no existe
    elif err.errno == errorcode.ER_BAD_DB_ERROR:
        print("La base de datos no existe.") #
    # Para cualquier otro tipo de error
    else:
        print(err) #
        print("Código de Error:", err.errno) #
        print("SQLSTATE", err.sqlstate) #
        print("Mensaje", err.msg) #

In [ ]:
mycursor = cnx.cursor() # Iniciamos el cursor
mycursor.execute("CREATE SCHEMA IF NOT EXISTS ---------- ") # creamos schema AQUI PONER EL NOMRE QUE ELEGIMO
mycursor.execute("USE ---------") #  NOMBRE QUE HEMOS ELEGIDOnos movemos al schema que acabamos de crear
# abrimos el cursor asociado a esta conexión

In [ ]:
mycursor = cnx.cursor() 
# abrimos el cursor asociado a esta conexión

CREAMOS LAS TABLAS


In [ ]:
tabla_empleados = #aqui va el codigo de msq, solo copiarlo.





mycursor.execute(tabla_empleados)
cnx.commit()
print("Tabla creada correctamente")

In [ ]:
tabla_satisfaccion = #copia codigo msql




mycursor.execute(tabla_satisfaccion)
cnx.commit()
print("Tabla creada correctamente")

In [ ]:
tabla_evaluacion= #copia codigo msql




mycursor.execute(tabla_evaluacion)
cnx.commit()
print("Tabla creada correctamente")

In [ ]:
tabla_contrato = #copia codigo msql



mycursor.execute(tabla_contrato)
cnx.commit()
print("Tabla creada correctamente")

In [ ]:
tabla_trayectoria = #copia codigo msql




mycursor.execute(tabla_trayectoria)
cnx.commit()
print("Tabla creada correctamente")

In [ ]:
tabla_salario = #copia codigo msql




mycursor.execute(tabla_salario)
cnx.commit()
print("Tabla creada correctamente")

In [ ]:
import csv
import mysql.connector

def ejecutar_etl_rrhh(archivo_csv, cnx):
    cursor = cnx.cursor()
    
    with open(hr.csv, newline='', encoding='utf-8') as f:
        # El DictReader usa la primera fila del CSV como llaves del diccionario
        reader = csv.DictReader(f)
        
        for fila in reader:
            try:
                # --- PASO 1: TRANSFORMACIÓN ---
                # Extraemos y limpiamos los datos según tus tablas
                emp_id = int(fila['EmployeeNumber'])
                
                # Datos para tabla 'empleado' (RAÍZ)
                data_empleado = (emp_id, int(fila['Age']), fila['Gender'], fila['MaritalStatus'], int(fila['Education']))
                
                # Datos para tabla 'contrato'
                data_contrato = (emp_id, fila['Department'], fila['JobRole'], int(fila['JobLevel']), fila['OverTime'])
                
                # Datos para tabla 'salario'
                data_salario = (emp_id, int(fila['MonthlyIncome']), int(fila['PercentSalaryHike']))

                # --- PASO 2: CARGA (LOAD) ---
                
                # A. Insertar en Maestro (Siempre primero)
                cursor.execute("""
                    INSERT INTO empleado (employee_number, age, gender, marital_status, education)
                    VALUES (%s, %s, %s, %s, %s)
                    ON DUPLICATE KEY UPDATE age = VALUES(age)
                """, data_empleado)

                # B. Insertar en Contrato
                cursor.execute("""
                    INSERT INTO contrato (employee_number, departament, job_role, job_level, over_time)
                    VALUES (%s, %s, %s, %s, %s)
                    ON DUPLICATE KEY UPDATE job_role = VALUES(job_role)
                """, data_contrato)

                # C. Insertar en Salario
                cursor.execute("""
                    INSERT INTO salario (employee_number, monthly_income, percent_salary_hike)
                    VALUES (%s, %s, %s)
                    ON DUPLICATE KEY UPDATE monthly_income = VALUES(monthly_income)
                """, data_salario)

                # --- PASO 3: CONFIRMACIÓN ---
                # Si todas las inserciones de este empleado fueron exitosas:
                cnx.commit()
                
            except Exception as e:
                # Si algo falla en CUALQUIER tabla, deshacemos los cambios de este empleado
                cnx.rollback()
                print(f"Error procesando empleado {fila.get('EmployeeNumber')}: {e}")

# --- BLOQUE PRINCIPAL ---
config = {
    'user': 'root',
    'password': 'tu_password',
    'host': '127.0.0.1',
    'database': 'nombre_tu_bd'
}

try:
    conexion = mysql.connector.connect(**config)
    ejecutar_etl_rrhh('tu_archivo_datos.csv', conexion)
    print("ETL Finalizado exitosamente.")
finally:
    if conexion.is_connected():
        conexion.close()